# Document-Aware Extraction with Llama Vision

Document-aware extraction notebook for processing invoices, receipts, and bank statements using Llama-3.2-Vision-Instruct.

**Key Features:**
- Document type detection - Automatically identifies invoice, receipt, or bank statement
- YAML-based prompt configuration - Static prompts for each document type
- V100 GPU optimization - ResilientGenerator with 6-tier OOM fallback system
- Universal field extraction - Consistent fields across all document types
- Memory management - Real-time fragmentation detection and cleanup
- Rich console interface - Beautiful formatting and comprehensive status reporting
- Production-ready error handling - Emergency model reload and CPU fallback
- Ground truth validation - Automated accuracy assessment and performance metrics

**Document Types Supported:**
- **Invoices** - Bills, quotes, estimates with line items and totals
- **Receipts** - Purchase confirmations with items and payment details
- **Bank Statements** - Transaction tables with dates and amounts

**Modern Architecture:**
- Modular V100-optimized extractor class
- External configuration files (YAML)
- Comprehensive memory monitoring
- Advanced error recovery systems

# Core Library Imports

Essential imports for Vision Language Model processing, image handling, and data manipulation.

In [ ]:
# Standard library imports
import warnings
from pathlib import Path

# Third-party imports
import pandas as pd
import torch
import yaml
from PIL import Image
from rich import print as rprint
from rich.console import Console
from rich.syntax import Syntax
from rich.table import Table
from transformers import AutoProcessor, MllamaForConditionalGeneration

# Local imports - GPU optimization utilities
from common.gpu_optimization import (
    comprehensive_memory_cleanup,
    configure_cuda_memory_allocation,
    optimize_model_for_v100,
)

warnings.filterwarnings('ignore')

rprint("[bold green]✅ Core libraries imported successfully[/bold green]")

In [ ]:
# Configuration for document type detection and extraction
DETECTION_PROMPT_FILE = "prompts/document_type_detection.yaml"  # Document type detection prompts
DETECTION_PROMPT_KEY = "detection"  # Which detection prompt to use

# Extraction prompt files for each document type
PROMPT_FILES = {
    "INVOICE": "prompts/invoice_extraction.yaml",
    "RECEIPT": "prompts/receipt_extraction.yaml",
    "BANK_STATEMENT": "prompts/bank_statement_extraction_original.yaml",
    "STATEMENT": "prompts/bank_statement_extraction_original.yaml"  # Alias
}

# Default prompt key for each document type
PROMPT_KEYS = {
    "INVOICE": "standard",
    "RECEIPT": "standard", 
    "BANK_STATEMENT": "flat",
    "STATEMENT": "flat"
}

# Configuration for model
MODEL_PATH = "/home/jovyan/nfs_share/models/Llama-3.2-11B-Vision-Instruct"

# Test images - can be any document type
TEST_IMAGE = "evaluation_data/commbank_flat_simple.png"  # Bank statement
# TEST_IMAGE = "evaluation_data/image_001.png"  # Invoice
# TEST_IMAGE = "evaluation_data/image_002.png"  # Receipt
# TEST_IMAGE = "evaluation_data/commbank_flat_complex.png"  # Bank statement
# TEST_IMAGE = "evaluation_data/anz_statement_001.png"  # Bank statement

# Ground truth CSV for evaluation
GROUND_TRUTH_CSV = "evaluation_data/ground_truth.csv"

# V100 PRODUCTION CONFIGURATION - MODERN APPROACH
USE_QUANTIZATION = True   # Enable BitsAndBytesConfig (modern approach)
DEVICE_MAP = "auto"       # Automatic device mapping
MAX_NEW_TOKENS = 4000     # V100 OPTIMIZED - Prevents OOM
TORCH_DTYPE = "bfloat16"  # V100 COMPATIBLE - More efficient
LOW_CPU_MEM_USAGE = True  # MEMORY EFFICIENT - Essential for V100

# Initialize Rich console
console = Console()
rprint("[bold blue]🚀 Document-Aware Configuration loaded[/bold blue]")
rprint("[yellow]⚠️ V100 Production Mode: Modern BitsAndBytesConfig approach[/yellow]")
rprint(f"[cyan]🎯 TEST IMAGE: {TEST_IMAGE}[/cyan]")
rprint("[green]💡 Document type will be auto-detected[/green]")

# Validate the selected image exists
if Path(TEST_IMAGE).exists():
    rprint(f"[green]✅ Test image found: {Path(TEST_IMAGE).name}[/green]")
else:
    rprint(f"[red]❌ Test image not found: {TEST_IMAGE}[/red]")
    rprint("[yellow]💡 Update TEST_IMAGE path above[/yellow]")

# Image Selection & Validation

Select and validate the bank statement image for processing, with fallback to available images.

In [ ]:
# =============================================================================
# DOCUMENT IMAGE SELECTION & VALIDATION
# =============================================================================

rprint("[bold blue]🎯 Using document image selection...[/bold blue]")

# Use the specified test image
IMAGE_PATH = TEST_IMAGE

if IMAGE_PATH:
    # Validate selected image exists
    image_path = Path(IMAGE_PATH)
    if image_path.exists():
        rprint(f"[bold green]🎉 Document ready: {image_path.name}[/bold green]")
        
        # Display image info
        try:
            from PIL import Image
            with Image.open(image_path) as img:
                width, height = img.size
                format_info = img.format or "Unknown"
                
                image_info_table = Table(title="📊 Document Image Information", border_style="green")
                image_info_table.add_column("Property", style="cyan")
                image_info_table.add_column("Value", style="yellow")
                
                image_info_table.add_row("Filename", image_path.name)
                image_info_table.add_row("Format", format_info)
                image_info_table.add_row("Dimensions", f"{width} × {height} pixels")
                image_info_table.add_row("File Size", f"{image_path.stat().st_size / 1024:.1f} KB")
                image_info_table.add_row("Full Path", str(image_path))
                image_info_table.add_row("Document Type", "To be detected...")
                
                console.print(image_info_table)
                
        except Exception as e:
            rprint(f"[yellow]⚠️ Could not read image info: {e}[/yellow]")
    else:
        rprint(f"[red]❌ Image file does not exist: {image_path}[/red]")
        rprint("[yellow]💡 Update TEST_IMAGE in configuration cell[/yellow]")
        IMAGE_PATH = None
else:
    rprint("[red]❌ No document image specified[/red]")

console.rule("[bold blue]Document Selection Complete[/bold blue]")

# Model Loading & Validation

Load the Llama 3.2 Vision model with comprehensive validation and GPU optimization for V100.

In [ ]:
# =============================================================================
# MODEL LOADING & VALIDATION - V100 PRODUCTION OPTIMIZED
# =============================================================================

# Import V100 optimization utilities

rprint("[bold yellow]🚀 Loading Llama Vision model with V100 production optimizations...[/bold yellow]")

try:
    # PHASE 1: Configure V100-specific CUDA memory allocation
    rprint("[blue]🔧 Configuring V100-optimized CUDA memory allocation...[/blue]")
    configure_cuda_memory_allocation()  # Sets 32MB blocks for V100
    
    # Validate model path first
    if not Path(MODEL_PATH).exists():
        raise FileNotFoundError(f"Model path does not exist: {MODEL_PATH}")
    
    # PHASE 2: Configure V100-optimized quantization (CORRECTED)
    if USE_QUANTIZATION:  # ✅ FIXED: Removed undefined LOAD_IN_8BIT reference
        rprint("[yellow]🔧 Configuring V100-optimized 8-bit quantization with BitsAndBytesConfig[/yellow]")
        
        from transformers import BitsAndBytesConfig
        
        # V100-OPTIMIZED quantization configuration (from V100_MEMORY_STRATEGIES.md)
        quantization_config = BitsAndBytesConfig(
            load_in_8bit=True,
            llm_int8_enable_fp32_cpu_offload=True,  # ✅ CRITICAL for V100 memory management
            llm_int8_skip_modules=["vision_tower", "multi_modal_projector"],  # ✅ Skip vision components
            llm_int8_threshold=6.0,  # Standard threshold for outlier detection
        )
        
        rprint("[green]✅ V100-optimized BitsAndBytesConfig configured[/green]")
        rprint("[cyan]💡 Key V100 optimizations:[/cyan]")
        rprint("[cyan]   • CPU offload enabled for memory efficiency[/cyan]")
        rprint("[cyan]   • Vision modules skipped to prevent quantization issues[/cyan]")
        rprint("[cyan]   • 32MB CUDA memory blocks configured[/cyan]")
    else:
        quantization_config = None
        rprint("[blue]ℹ️ No quantization - loading in full precision[/blue]")
    
    # PHASE 3: Load model with V100-optimized configuration
    rprint("[dim]Loading Llama-3.2-Vision model with V100 optimizations...[/dim]")
    model = MllamaForConditionalGeneration.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.bfloat16,
        device_map=DEVICE_MAP,
        quantization_config=quantization_config,  # V100-optimized config
        low_cpu_mem_usage=True,  # Memory optimization
    )
    
    # Load processor
    rprint("[dim]Loading processor...[/dim]")
    processor = AutoProcessor.from_pretrained(MODEL_PATH)
    
    # PHASE 4: Apply V100-specific model optimizations
    from common.gpu_optimization import optimize_model_for_v100
    optimize_model_for_v100(model)
    
    # Comprehensive model validation
    if model is None:
        raise RuntimeError("Model failed to load - returned None")
    if processor is None:
        raise RuntimeError("Processor failed to load - returned None")
    
    # Model parameter analysis
    model_params = sum(p.numel() for p in model.parameters())
    if model_params < 1e9:  # Less than 1B parameters seems wrong
        rprint(f"[yellow]⚠️ Warning: Model has unusually few parameters: {model_params:,.0f}[/yellow]")
    
    # Device placement validation
    model_device = next(model.parameters()).device
    if model_device.type == 'cpu' and torch.cuda.is_available():
        rprint("[yellow]⚠️ Warning: Model loaded on CPU despite CUDA availability[/yellow]")
    
    rprint("[bold green]✅ Model and processor loaded successfully![/bold green]")
    rprint(f"[cyan]📊 Device: {model_device}[/cyan]")
    
    # PHASE 5: V100-specific memory analysis and validation
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name()
        memory_allocated = torch.cuda.memory_allocated() / 1e9
        memory_reserved = torch.cuda.memory_reserved() / 1e9
        memory_total = torch.cuda.get_device_properties(0).total_memory / 1e9
        
        rprint(f"[magenta]🎮 GPU: {gpu_name}[/magenta]")
        rprint(f"[blue]💾 Memory Allocated: {memory_allocated:.2f}GB[/blue]")
        rprint(f"[blue]💾 Memory Reserved: {memory_reserved:.2f}GB[/blue]")
        rprint(f"[blue]💾 Total GPU Memory: {memory_total:.0f}GB[/blue]")
        
        # V100-specific memory warnings and fragmentation detection
        memory_usage_pct = memory_allocated / memory_total * 100
        fragmentation = memory_reserved - memory_allocated
        
        if "V100" in gpu_name:
            rprint("[blue]🎯 Detected V100: Applying specialized memory management[/blue]")
            if fragmentation > 1.0:
                rprint(f"[red]🚨 FRAGMENTATION DETECTED: {fragmentation:.2f}GB gap[/red]")
                rprint("[yellow]⚠️ Memory pool may need cleanup between operations[/yellow]")
            
            if memory_usage_pct > 80:
                rprint(f"[red]🚨 CRITICAL V100 memory usage: {memory_usage_pct:.1f}% - REDUCE MAX_NEW_TOKENS[/red]")
            elif memory_usage_pct > 60:
                rprint(f"[yellow]⚠️ HIGH V100 memory usage: {memory_usage_pct:.1f}% - Monitor closely[/yellow]")
            elif memory_usage_pct > 40:
                rprint(f"[yellow]⚠️ Moderate V100 memory usage: {memory_usage_pct:.1f}%[/yellow]")
            else:
                rprint(f"[green]✅ Excellent V100 memory usage: {memory_usage_pct:.1f}%[/green]")
        else:
            # H200 or other high-memory GPU
            if memory_usage_pct > 70:
                rprint(f"[red]⚠️ High GPU memory usage: {memory_usage_pct:.1f}%[/red]")
            elif memory_usage_pct > 50:
                rprint(f"[yellow]⚠️ Moderate GPU memory usage: {memory_usage_pct:.1f}%[/yellow]")
            else:
                rprint(f"[green]✅ Good GPU memory usage: {memory_usage_pct:.1f}%[/green]")
    else:
        rprint("[yellow]⚠️ CUDA not available - using CPU (significantly slower)[/yellow]")
    
    # PHASE 6: Comprehensive V100 configuration validation table
    config_table = Table(title="🔧 V100 Production Model Configuration", border_style="blue")
    config_table.add_column("Setting", style="cyan", no_wrap=True)
    config_table.add_column("Value", style="yellow")
    config_table.add_column("V100 Status", style="green")
    
    # V100-specific configuration details
    config_table.add_row("Model Path", MODEL_PATH, "✅ Valid")
    config_table.add_row("Device Placement", str(model_device), "✅ Loaded")
    config_table.add_row("Quantization Method", 
                        "V100-optimized BitsAndBytesConfig" if USE_QUANTIZATION else "Disabled", 
                        "✅ V100 Optimized" if USE_QUANTIZATION else "✅ Full Precision")
    config_table.add_row("CPU Offload", "Enabled" if USE_QUANTIZATION else "N/A", 
                        "✅ V100 Memory Efficient" if USE_QUANTIZATION else "N/A")
    config_table.add_row("Vision Skip Modules", 
                        "vision_tower, multi_modal_projector" if USE_QUANTIZATION else "N/A",
                        "✅ V100 Compatible" if USE_QUANTIZATION else "N/A")
    config_table.add_row("Max New Tokens", str(MAX_NEW_TOKENS), 
                        "✅ V100 Safe" if MAX_NEW_TOKENS <= 4000 else "⚠️ High for V100")
    config_table.add_row("Model Parameters", f"{model_params:,.0f}", "✅ Loaded")
    config_table.add_row("CUDA Memory Blocks", "32MB (V100 optimized)" if "V100" in torch.cuda.get_device_name() else "64MB (Standard)",
                        "✅ Fragmentation resistant")
    config_table.add_row("Memory Optimization", "V100 Enhanced", "✅ Production ready")
    
    console.print(config_table)
    
    # Model functionality test
    rprint("[dim]Running model compatibility test...[/dim]")
    try:
        # Test chat template processing
        test_messages = [
            {"role": "user", "content": [
                {"type": "text", "text": "Test"}
            ]}
        ]
        test_input_text = processor.apply_chat_template(test_messages, add_generation_prompt=True)
        
        if len(test_input_text) < 10:
            raise RuntimeError("Chat template processing failed")
        
        rprint("[green]✅ Model compatibility test passed[/green]")
        
    except Exception as e:
        rprint(f"[red]❌ Model compatibility test failed: {e}[/red]")
        raise RuntimeError(f"Model validation failed: {e}") from e

    # PHASE 7: Initial comprehensive cleanup to establish clean state
    rprint("[dim]Performing initial V100 memory cleanup...[/dim]")
    comprehensive_memory_cleanup(model, processor)

except Exception as e:
    rprint("[bold red]❌ CRITICAL ERROR: V100-optimized model loading failed[/bold red]")
    rprint(f"[red]💥 Error: {e}[/red]")
    rprint("[yellow]💡 Check model path, GPU memory, and V100 compatibility requirements[/yellow]")
    raise  # Re-raise to prevent notebook execution with invalid model

rprint("[bold green]🎉 V100-optimized model loading and validation complete![/bold green]")
rprint("[cyan]🔧 V100 optimizations active: CPU offload, vision skip, 32MB blocks[/cyan]")

# Prompt Engineering

Define the optimized prompt for bank statement transaction extraction with clear visual guidance.

In [ ]:
# =============================================================================
# DOCUMENT TYPE DETECTION
# =============================================================================

rprint("[bold blue]📋 Document Type Detection[/bold blue]")

# Load detection prompt from YAML
try:
    detection_path = Path(DETECTION_PROMPT_FILE)
    
    if not detection_path.exists():
        rprint(f"[red]❌ Detection prompt file not found at {DETECTION_PROMPT_FILE}[/red]")
        raise FileNotFoundError(f"Required prompt file not found: {DETECTION_PROMPT_FILE}")
    
    # Load the YAML file
    with open(detection_path, 'r') as f:
        detection_config = yaml.safe_load(f)
    
    # Extract the detection prompt
    if DETECTION_PROMPT_KEY in detection_config.get("prompts", {}):
        detection_prompt_data = detection_config["prompts"][DETECTION_PROMPT_KEY]
        DETECTION_PROMPT = detection_prompt_data["prompt"]
        detection_name = detection_prompt_data.get("name", DETECTION_PROMPT_KEY)
        
        rprint(f"[green]✅ Loaded detection prompt: {detection_name}[/green]")
    else:
        available_keys = list(detection_config.get('prompts', {}).keys())
        rprint(f"[red]❌ Detection prompt key '{DETECTION_PROMPT_KEY}' not found[/red]")
        raise KeyError(f"Prompt '{DETECTION_PROMPT_KEY}' not found. Available: {available_keys}")
    
    # Get type mappings for normalization
    TYPE_MAPPINGS = detection_config.get("type_mappings", {})
    
except Exception as e:
    rprint(f"[red]❌ Error loading detection prompt: {e}[/red]")
    raise

# Detect document type using the extractor
if IMAGE_PATH:
    rprint(f"[yellow]🔍 Detecting document type for: {Path(IMAGE_PATH).name}[/yellow]")
    
    try:
        # Use the extractor to run detection prompt
        detection_result = extractor.test_extraction(
            IMAGE_PATH, 
            DETECTION_PROMPT,
            max_new_tokens=50  # Short response for detection
        )
        
        if detection_result.get("success"):
            raw_detection = detection_result["raw_result"]["raw_response"]
            
            # Parse the detection response
            detected_type = raw_detection.strip().upper()
            
            # Apply type mappings to normalize variations
            for variation, canonical in TYPE_MAPPINGS.items():
                if variation.lower() in raw_detection.lower():
                    detected_type = canonical
                    break
            
            # Ensure we have a valid document type
            if detected_type not in ["INVOICE", "RECEIPT", "BANK_STATEMENT", "STATEMENT"]:
                # Try to extract from response
                if "INVOICE" in detected_type or "BILL" in detected_type:
                    detected_type = "INVOICE"
                elif "RECEIPT" in detected_type:
                    detected_type = "RECEIPT"
                elif "STATEMENT" in detected_type or "BANK" in detected_type:
                    detected_type = "BANK_STATEMENT"
                else:
                    rprint(f"[yellow]⚠️ Unknown document type: {detected_type}, defaulting to INVOICE[/yellow]")
                    detected_type = "INVOICE"
            
            # Normalize STATEMENT to BANK_STATEMENT
            if detected_type == "STATEMENT":
                detected_type = "BANK_STATEMENT"
            
            DOCUMENT_TYPE = detected_type
            
            # Display detection result
            detection_table = Table(title="🔍 Document Type Detection Result", border_style="blue")
            detection_table.add_column("Property", style="cyan")
            detection_table.add_column("Value", style="yellow")
            
            detection_table.add_row("Detected Type", DOCUMENT_TYPE)
            detection_table.add_row("Prompt File", PROMPT_FILES.get(DOCUMENT_TYPE, "Unknown"))
            detection_table.add_row("Prompt Key", PROMPT_KEYS.get(DOCUMENT_TYPE, "standard"))
            detection_table.add_row("Raw Response", raw_detection[:100])
            
            console.print(detection_table)
            
            rprint(f"[bold green]✅ Document type detected: {DOCUMENT_TYPE}[/bold green]")
            
        else:
            rprint(f"[red]❌ Detection failed: {detection_result.get('error', 'Unknown error')}[/red]")
            rprint("[yellow]⚠️ Defaulting to INVOICE[/yellow]")
            DOCUMENT_TYPE = "INVOICE"
            
    except Exception as e:
        rprint(f"[red]❌ Error during detection: {e}[/red]")
        rprint("[yellow]⚠️ Defaulting to INVOICE[/yellow]")
        DOCUMENT_TYPE = "INVOICE"
else:
    rprint("[red]❌ No image path available for detection[/red]")
    DOCUMENT_TYPE = "INVOICE"

console.rule("[bold green]Document Type Detection Complete[/bold green]")

# Document Type Detection

Detect the type of document (invoice, receipt, or bank statement) before selecting the appropriate extraction prompt.

In [ ]:
# ============================================================================= 
# DOCUMENT-SPECIFIC PROMPT LOADING FROM YAML
# =============================================================================
# Initialize Rich console if not already done
if 'console' not in globals():
    console = Console()

rprint("[bold blue]📋 Loading Document-Specific Prompt from YAML[/bold blue]")

# Select prompt file based on detected document type
PROMPT_FILE = PROMPT_FILES.get(DOCUMENT_TYPE, "prompts/invoice_extraction.yaml")
PROMPT_KEY = PROMPT_KEYS.get(DOCUMENT_TYPE, "standard")

rprint(f"[cyan]📁 Document type: {DOCUMENT_TYPE}[/cyan]")
rprint(f"[cyan]📁 Prompt file: {PROMPT_FILE}[/cyan]")
rprint(f"[cyan]🔑 Selected prompt: {PROMPT_KEY}[/cyan]")

# Load prompts from YAML file
try:
    prompt_path = Path(PROMPT_FILE)
    
    if not prompt_path.exists():
        rprint(f"[red]❌ Prompt file not found at {PROMPT_FILE}[/red]")
        rprint(f"[yellow]💡 Please ensure the YAML file exists at: {prompt_path.absolute()}[/yellow]")
        raise FileNotFoundError(f"Required prompt file not found: {PROMPT_FILE}")
    
    # Load the YAML file
    with open(prompt_path, 'r') as f:
        prompt_config = yaml.safe_load(f)
    
    # Extract the selected prompt
    if PROMPT_KEY in prompt_config.get("prompts", {}):
        prompt_data = prompt_config["prompts"][PROMPT_KEY]
        EXTRACTION_PROMPT = prompt_data["prompt"]
        prompt_name = prompt_data.get("name", PROMPT_KEY)
        prompt_description = prompt_data.get("description", "")
        
        rprint(f"[green]✅ Loaded prompt: {prompt_name}[/green]")
        if prompt_description:
            rprint(f"[dim]{prompt_description}[/dim]")
    else:
        available_keys = list(prompt_config.get('prompts', {}).keys())
        rprint(f"[red]❌ Prompt key '{PROMPT_KEY}' not found in {PROMPT_FILE}[/red]")
        rprint(f"[yellow]Available prompts: {', '.join(available_keys)}[/yellow]")
        raise KeyError(f"Prompt '{PROMPT_KEY}' not found. Available: {available_keys}")
    
    # Load settings if available
    settings = prompt_config.get("settings", {})
    if settings:
        rprint("[dim]📊 Loaded settings from YAML:[/dim]")
        for key, value in settings.items():
            rprint(f"[dim]  • {key}: {value}[/dim]")
    
except FileNotFoundError as e:
    rprint(f"[red]❌ CRITICAL: {e}[/red]")
    rprint("[yellow]Please create the prompt YAML file before running this notebook.[/yellow]")
    raise
    
except yaml.YAMLError as e:
    rprint(f"[red]❌ Error parsing YAML file: {e}[/red]")
    rprint("[yellow]Please check the YAML syntax in {PROMPT_FILE}[/yellow]")
    raise
    
except Exception as e:
    rprint(f"[red]❌ Error loading prompt from YAML: {e}[/red]")
    raise

# Display the loaded prompt with Rich formatting
console.rule("[bold blue]Loaded Extraction Prompt Display[/bold blue]")

# Display as syntax-highlighted markdown
rprint(f"[bold cyan]📋 {prompt_name} for {DOCUMENT_TYPE}:[/bold cyan]")
if prompt_description:
    rprint(f"[dim]{prompt_description}[/dim]")
syntax = Syntax(EXTRACTION_PROMPT[:500] + "...", "markdown", theme="monokai", line_numbers=True)
console.print(syntax)

# Display prompt statistics
prompt_lines = EXTRACTION_PROMPT.strip().split('\n')
prompt_words = len(EXTRACTION_PROMPT.split())
prompt_chars = len(EXTRACTION_PROMPT)

stats_table = Table(title="📊 Prompt Statistics", border_style="green")
stats_table.add_column("Metric", style="cyan")
stats_table.add_column("Value", style="yellow")

stats_table.add_row("Document Type", DOCUMENT_TYPE)
stats_table.add_row("Lines", str(len(prompt_lines)))
stats_table.add_row("Words", str(prompt_words))
stats_table.add_row("Characters", str(prompt_chars))
stats_table.add_row("Source", str(prompt_path))
stats_table.add_row("Prompt Key", PROMPT_KEY)

console.print(stats_table)

console.rule("[bold green]Prompt Loading Complete[/bold green]")

In [ ]:
from common.ground_truth_helpers import get_expected_transaction_count
from common.llama_vision_table_extractor import LlamaVisionTableExtractor

rprint("[bold green]✅ V100-optimized modules imported successfully[/bold green]")

In [ ]:
# =============================================================================
# V100 PRODUCTION INITIALIZATION
# =============================================================================

# Initialize the V100-optimized extractor with existing model and processor
rprint("[bold green]🚀 Initializing V100-optimized LlamaVisionTableExtractor...[/bold green]")

try:
    extractor = LlamaVisionTableExtractor(processor=processor, model=model)
    
    # Validate V100 optimization status
    if hasattr(extractor, 'resilient_generator'):
        rprint("[green]✅ ResilientGenerator initialized for V100 OOM protection[/green]")
    else:
        rprint("[red]❌ ResilientGenerator initialization failed[/red]")
        
    rprint("[green]✅ V100-optimized extractor ready for production use![/green]")
    rprint("[cyan]🎯 V100 Features Active:[/cyan]")
    rprint("[cyan]   • ResilientGenerator with 6-tier OOM fallback[/cyan]")
    rprint("[cyan]   • Memory fragmentation detection and cleanup[/cyan]") 
    rprint("[cyan]   • Individual processing for memory stability[/cyan]")
    rprint("[cyan]   • Emergency model reload capabilities[/cyan]")
    rprint("[cyan]   • CPU fallback as ultimate safety net[/cyan]")
    
except Exception as e:
    rprint(f"[red]❌ CRITICAL: V100 extractor initialization failed: {e}[/red]")
    raise

In [ ]:
# =============================================================================
# V100 PRODUCTION DEMONSTRATION - DOCUMENT-AWARE EXTRACTION
# =============================================================================

rprint("[bold cyan]🧪 Testing V100-optimized LlamaVisionTableExtractor with document-aware extraction...[/bold cyan]")

console.rule("[bold blue]V100 Production Document-Aware Demonstration[/bold blue]")

# Test with the current image and document-specific prompt
if IMAGE_PATH:
    rprint(f"[yellow]🎯 Running comprehensive V100 production test on {DOCUMENT_TYPE}...[/yellow]")
    
    # Use the enhanced test_extraction method which includes V100 memory monitoring
    test_result = extractor.test_extraction(IMAGE_PATH, EXTRACTION_PROMPT)
    
    # Display comprehensive test results
    if test_result.get("success"):
        result = test_result["raw_result"]
        
        rprint(f"[cyan]📄 Image processed:[/cyan] {result['image_path']}")
        rprint(f"[cyan]📋 Document type:[/cyan] {DOCUMENT_TYPE}")
        
        # Document-specific result handling
        if DOCUMENT_TYPE == "BANK_STATEMENT":
            rprint(f"[magenta]📊 Transactions found:[/magenta] {result.get('transaction_count', 'N/A')}")
        else:
            # For invoices/receipts, count line items
            if result.get('parsed_data'):
                parsed_fields = result['parsed_data']
                if isinstance(parsed_fields, dict):
                    line_items = parsed_fields.get('LINE_ITEM_DESCRIPTIONS', '')
                    if line_items and line_items != 'NOT_FOUND':
                        item_count = len(line_items.split(' | '))
                        rprint(f"[magenta]📊 Line items found:[/magenta] {item_count}")
        
        rprint(f"[green]⏰ Processing time:[/green] {test_result['processing_time']:.3f}s")
        rprint(f"[blue]🚀 V100 Optimized:[/blue] {result.get('v100_optimized', True)}")
        
        # V100-specific metrics
        if "memory_metrics" in test_result:
            memory_metrics = test_result["memory_metrics"]
            rprint(f"[yellow]💾 Memory delta:[/yellow] {memory_metrics['memory_delta_gb']:+.3f}GB")
            rprint(f"[yellow]📊 Fragmentation change:[/yellow] {memory_metrics['fragmentation_change_gb']:+.3f}GB")
            rprint(f"[green]🛡️ ResilientGenerator:[/green] {memory_metrics['resilient_generator_used']}")
        
        # Display extracted fields based on document type
        if DOCUMENT_TYPE == "BANK_STATEMENT" and result.get('bank_details'):
            # Bank statement specific display
            bank_details = result['bank_details']
            bank_details_table = Table(title="🏦 V100-Extracted Bank Account Details", border_style="blue")
            bank_details_table.add_column("Field", style="cyan")
            bank_details_table.add_column("Value", style="yellow")
            
            for field, value in bank_details.items():
                if value != 'NOT_FOUND':  # Only show fields that have values
                    bank_details_table.add_row(field, value)
            
            console.print(bank_details_table)
            
            # Show parsed transaction data in a table
            if result['parsed_data']:
                parsed_table = Table(title="📋 V100-Extracted Transaction Data", border_style="green")
                parsed_table.add_column("#", style="dim", width=3)
                parsed_table.add_column("Date", style="cyan")
                parsed_table.add_column("Description", style="white", max_width=30)
                parsed_table.add_column("Debit", style="red")
                parsed_table.add_column("Credit", style="green")
                parsed_table.add_column("Balance", style="yellow")
                
                for i, transaction in enumerate(result['parsed_data'], 1):
                    parsed_table.add_row(
                        str(i),
                        transaction.get('date', ''),
                        transaction.get('description', ''),
                        transaction.get('debit', ''),
                        transaction.get('credit', ''),
                        transaction.get('balance', '')
                    )
                
                console.print(parsed_table)
        else:
            # Invoice/Receipt display - show extracted fields
            raw_response = result.get('raw_response', '')
            if raw_response:
                # Parse the response to extract fields
                field_table = Table(title=f"📋 V100-Extracted {DOCUMENT_TYPE} Fields", border_style="green")
                field_table.add_column("Field", style="cyan", width=25)
                field_table.add_column("Value", style="yellow", max_width=50)
                
                # Parse each line looking for field:value pairs
                lines = raw_response.split('\n')
                for line in lines:
                    if ':' in line and not line.startswith('#') and not line.startswith('|'):
                        parts = line.split(':', 1)
                        if len(parts) == 2:
                            field = parts[0].strip()
                            value = parts[1].strip()
                            if value and value != 'NOT_FOUND':
                                # Truncate long values for display
                                display_value = value[:48] + "..." if len(value) > 48 else value
                                field_table.add_row(field, display_value)
                
                console.print(field_table)
        
        # Get expected count based on document type
        if DOCUMENT_TYPE == "BANK_STATEMENT":
            expected_count = get_expected_transaction_count(IMAGE_PATH, GROUND_TRUTH_CSV)
            extracted_count = test_result.get('extracted_count', 0)
        else:
            # For invoices/receipts, count fields extracted
            expected_count = 14  # Standard field count for invoice/receipt
            extracted_count = len([1 for line in result.get('raw_response', '').split('\n') 
                                  if ':' in line and 'NOT_FOUND' not in line])
        
        # V100 Performance Comparison Table
        performance_table = Table(title=f"🚀 V100 Production Performance Analysis - {DOCUMENT_TYPE}", border_style="blue")
        performance_table.add_column("Metric", style="cyan")
        performance_table.add_column("Value", style="magenta")
        performance_table.add_column("V100 Status", style="green")
        
        # Performance metrics
        accuracy_pct = (extracted_count / expected_count * 100) if expected_count > 0 else 0
        performance_table.add_row("Document Type", DOCUMENT_TYPE, "✅ Detected")
        performance_table.add_row("Extraction Method", "V100 ResilientGenerator", "✅ Production Ready")
        
        if DOCUMENT_TYPE == "BANK_STATEMENT":
            performance_table.add_row("Transactions Extracted", str(extracted_count), 
                                     "✅ Perfect" if extracted_count == expected_count else f"⚠️ Expected {expected_count}")
        else:
            performance_table.add_row("Fields Extracted", str(extracted_count), 
                                     "✅ Good" if extracted_count >= 10 else "⚠️ Review")
        
        performance_table.add_row("Accuracy", f"{accuracy_pct:.1f}%", 
                                 "✅ Excellent" if accuracy_pct >= 95 else "⚠️ Needs Review")
        performance_table.add_row("Processing Speed", f"{test_result['processing_time']:.3f}s", 
                                 "✅ V100 Optimized")
        performance_table.add_row("Memory Management", "Active Monitoring", "✅ Fragmentation Detection")
        performance_table.add_row("OOM Protection", "6-Tier Fallback", "✅ ResilientGenerator")
        performance_table.add_row("Response Quality", f"{test_result['response_length']} chars", 
                                 "✅ Good" if 100 < test_result['response_length'] < 5000 else "⚠️ Review")
        performance_table.add_row("Hallucination Check", "Pattern Detection", 
                                 "❌ Detected" if test_result.get('hallucination_detected', False) else "✅ Clean")
        
        console.print(performance_table)
        
    else:
        rprint(f"[red]❌ V100 test failed: {test_result.get('error', 'Unknown error')}[/red]")

else:
    rprint("[red]❌ No image path available for V100 testing[/red]")

console.rule("[bold green]Testing Complete[/bold green]")

In [ ]:
# =============================================================================
# VISUAL COMPARISON - DOCUMENT IMAGE DISPLAY
# =============================================================================

from IPython.display import Image as IPImage
from IPython.display import display

if IMAGE_PATH and Path(IMAGE_PATH).exists():
    rprint(f"[cyan]📄 Displaying {DOCUMENT_TYPE} for visual comparison: {Path(IMAGE_PATH).name}[/cyan]")
    
    # Display the image
    display(IPImage(filename=IMAGE_PATH))
    
    rprint(f"[green]✅ Compare extraction results above with the visual {DOCUMENT_TYPE} image[/green]")
else:
    rprint(f"[red]❌ Document image not found: {IMAGE_PATH}[/red]")

# Field-Level Ground Truth Evaluation

Comprehensive field-by-field comparison between extracted data and ground truth values.

In [ ]:
# =============================================================================
# RESPONSE PREPROCESSING - Import from common module
# =============================================================================

from common.response_preprocessing import (
    clean_markdown_response,
    extract_statement_date_range,
    extract_transaction_data_from_table,
    map_bank_fields_to_universal,
    map_invoice_fields_to_universal,
    map_receipt_fields_to_universal,
    map_fields_to_universal,  # Universal mapping function
)

rprint("[green]✅ Response preprocessing functions imported from common module[/green]")

In [ ]:
# =============================================================================
# FIELD-LEVEL GROUND TRUTH EVALUATION - DOCUMENT-AWARE
# =============================================================================

from common.evaluation_metrics import calculate_field_accuracy
from common.extraction_parser import parse_extraction_response

rprint("[bold cyan]📊 FIELD-LEVEL GROUND TRUTH EVALUATION[/bold cyan]")
console.rule("[bold blue]Comprehensive Document-Aware Field Evaluation[/bold blue]")

if IMAGE_PATH and 'test_result' in globals() and test_result.get('success'):
    # Get the raw response from the test
    raw_response = test_result['raw_result']['raw_response']
    
    # CLEAN THE RESPONSE BEFORE PARSING
    cleaned_response = clean_markdown_response(raw_response)
    
    # MAP DOCUMENT-SPECIFIC FIELDS TO UNIVERSAL FIELD NAMES FOR EVALUATION
    mapped_response = map_fields_to_universal(cleaned_response, DOCUMENT_TYPE)
    
    image_filename = Path(IMAGE_PATH).name
    
    rprint(f"[yellow]🔍 Evaluating extraction for: {image_filename}[/yellow]")
    rprint(f"[dim]Document Type: {DOCUMENT_TYPE}[/dim]")
    rprint(f"[dim]Using {DOCUMENT_TYPE.lower()}-specific extraction with universal field mapping for evaluation[/dim]")
    
    # Parse the MAPPED extraction response to get field-level data
    extracted_fields = parse_extraction_response(mapped_response)
    
    # Load ground truth for this specific image
    try:
        ground_truth_df = pd.read_csv(GROUND_TRUTH_CSV)
        image_row = ground_truth_df[ground_truth_df['image_file'] == image_filename]
        
        if not image_row.empty:
            ground_truth = image_row.iloc[0].to_dict()
            
            rprint(f"[green]✅ Ground truth loaded for {image_filename}[/green]")
            rprint(f"[dim]Found {len([v for v in ground_truth.values() if v != 'NOT_FOUND' and pd.notna(v)])} ground truth fields[/dim]")
            
            # Create field comparison table
            comparison_table = Table(title=f"📋 Field-by-Field Comparison ({DOCUMENT_TYPE}→Universal Mapping)", border_style="green")
            comparison_table.add_column("Status", style="bold", width=8)
            comparison_table.add_column("Field", style="cyan", width=25)
            comparison_table.add_column("Extracted", style="yellow", max_width=40)
            comparison_table.add_column("Ground Truth", style="magenta", max_width=40)
            comparison_table.add_column("Score", style="blue", justify="right")
            
            # Track metrics
            total_fields = 0
            fields_found = 0
            exact_matches = 0
            partial_matches = 0
            field_scores = {}
            
            # Document type specific fields to focus on
            if DOCUMENT_TYPE == "BANK_STATEMENT":
                focus_fields = ["DOCUMENT_TYPE", "STATEMENT_DATE_RANGE", "LINE_ITEM_DESCRIPTIONS", 
                               "TRANSACTION_DATES", "TRANSACTION_AMOUNTS_PAID"]
            elif DOCUMENT_TYPE == "INVOICE":
                focus_fields = ["DOCUMENT_TYPE", "BUSINESS_ABN", "SUPPLIER_NAME", "INVOICE_DATE",
                               "LINE_ITEM_DESCRIPTIONS", "GST_AMOUNT", "TOTAL_AMOUNT"]
            elif DOCUMENT_TYPE == "RECEIPT":
                focus_fields = ["DOCUMENT_TYPE", "SUPPLIER_NAME", "INVOICE_DATE",
                               "LINE_ITEM_DESCRIPTIONS", "TOTAL_AMOUNT"]
            else:
                focus_fields = []
            
            # Compare each field
            for field_name in ground_truth.keys():
                if field_name == 'image_file':  # Skip the image filename column
                    continue
                    
                ground_value = ground_truth.get(field_name, 'NOT_FOUND')
                extracted_value = extracted_fields.get(field_name, 'NOT_FOUND')
                
                # Convert pandas NaN to NOT_FOUND
                if pd.isna(ground_value):
                    ground_value = 'NOT_FOUND'
                if pd.isna(extracted_value):
                    extracted_value = 'NOT_FOUND'
                    
                # Skip if both are NOT_FOUND (field not applicable)
                if str(ground_value) == 'NOT_FOUND' and str(extracted_value) == 'NOT_FOUND':
                    continue
                    
                total_fields += 1
                
                # Calculate field accuracy
                accuracy = calculate_field_accuracy(
                    extracted_value, 
                    ground_value, 
                    field_name,
                    debug=False
                )
                
                field_scores[field_name] = accuracy
                
                # Determine status
                if accuracy == 1.0:
                    status = "✅"
                    exact_matches += 1
                    if str(extracted_value) != 'NOT_FOUND':
                        fields_found += 1
                elif accuracy >= 0.8:
                    status = "≈"
                    partial_matches += 1
                    if str(extracted_value) != 'NOT_FOUND':
                        fields_found += 1
                else:
                    status = "❌"
                    if str(extracted_value) != 'NOT_FOUND':
                        fields_found += 1
                
                # Highlight focus fields for this document type
                if field_name in focus_fields:
                    field_name = f"🔑 {field_name}"
                
                # Truncate long values for display
                extracted_display = str(extracted_value)[:38] + ("..." if len(str(extracted_value)) > 38 else "")
                ground_display = str(ground_value)[:38] + ("..." if len(str(ground_value)) > 38 else "")
                
                # Add row to table
                comparison_table.add_row(
                    status,
                    field_name,
                    extracted_display,
                    ground_display,
                    f"{accuracy:.2f}"
                )
            
            # Display the comparison table
            console.print(comparison_table)
            
            # Calculate overall metrics
            overall_accuracy = sum(field_scores.values()) / len(field_scores) if field_scores else 0
            
            # Summary statistics table
            summary_table = Table(title=f"📈 Evaluation Summary ({DOCUMENT_TYPE} Extraction)", border_style="blue")
            summary_table.add_column("Metric", style="cyan")
            summary_table.add_column("Value", style="magenta")
            summary_table.add_column("Percentage", style="yellow")
            
            summary_table.add_row(
                "Document Type",
                DOCUMENT_TYPE,
                "-"
            )
            summary_table.add_row(
                "Total Fields Evaluated",
                str(total_fields),
                "100%"
            )
            summary_table.add_row(
                "Fields Found",
                str(fields_found),
                f"{(fields_found/total_fields*100):.1f}%" if total_fields > 0 else "0%"
            )
            summary_table.add_row(
                "Exact Matches",
                str(exact_matches),
                f"{(exact_matches/total_fields*100):.1f}%" if total_fields > 0 else "0%"
            )
            summary_table.add_row(
                "Partial Matches (≥0.8)",
                str(partial_matches),
                f"{(partial_matches/total_fields*100):.1f}%" if total_fields > 0 else "0%"
            )
            summary_table.add_row(
                "Overall Accuracy",
                f"{overall_accuracy:.3f}",
                f"{(overall_accuracy*100):.1f}%"
            )
            
            console.print(summary_table)
            
            # Document type detection accuracy
            if 'DOCUMENT_TYPE' in extracted_fields:
                extracted_doc_type = extracted_fields['DOCUMENT_TYPE']
                ground_doc_type = ground_truth.get('DOCUMENT_TYPE', 'NOT_FOUND')
                
                if extracted_doc_type != 'NOT_FOUND' and ground_doc_type != 'NOT_FOUND':
                    doc_type_match = extracted_doc_type.upper() == str(ground_doc_type).upper()
                    rprint("\n[bold]Document Type Detection:[/bold]")
                    rprint(f"  Extracted: {extracted_doc_type}")
                    rprint(f"  Ground Truth: {ground_doc_type}")
                    rprint(f"  Status: {'✅ Correct' if doc_type_match else '❌ Incorrect'}")
            
            # Performance assessment
            rprint("\n[bold cyan]📊 Performance Assessment:[/bold cyan]")
            rprint(f"[dim]Note: Using {DOCUMENT_TYPE.lower()}-specific extraction with field mapping[/dim]")
            if overall_accuracy >= 0.95:
                rprint("[bold green]🎉 EXCELLENT - Production Ready (≥95% accuracy)[/bold green]")
            elif overall_accuracy >= 0.85:
                rprint("[yellow]✅ GOOD - Minor improvements needed (85-94% accuracy)[/yellow]")
            elif overall_accuracy >= 0.75:
                rprint("[yellow]⚠️ FAIR - Significant improvements needed (75-84% accuracy)[/yellow]")
            else:
                rprint("[red]❌ POOR - Major improvements required (<75% accuracy)[/red]")
                
            # Processing performance
            if 'processing_time' in test_result:
                proc_time = test_result['processing_time']
                rprint(f"\n⏱️  Processing Time: {proc_time:.2f}s")
                if proc_time < 5:
                    rprint("[green]  Speed: Excellent (<5s)[/green]")
                elif proc_time < 10:
                    rprint("[yellow]  Speed: Good (5-10s)[/yellow]")
                else:
                    rprint("[red]  Speed: Needs optimization (>10s)[/red]")
                    
        else:
            rprint(f"[red]❌ No ground truth found for {image_filename}[/red]")
            rprint("[yellow]💡 Make sure the image filename matches exactly in ground_truth.csv[/yellow]")
            
    except Exception as e:
        rprint(f"[red]❌ Error loading ground truth: {e}[/red]")
        import traceback
        traceback.print_exc()
        
else:
    rprint("[yellow]⚠️ No extraction results available for evaluation[/yellow]")
    rprint("[dim]Run the extraction test in the cell above first[/dim]")

console.rule("[bold green]Field-Level Evaluation Complete[/bold green]")